<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day9_10_a_(260416)_React_Query_%EA%B8%B0%EB%B0%98_WMS_%EB%8C%80%EC%8B%9C%EB%B3%B4%EB%93%9C_%EC%84%9C%EB%B2%84_%EC%83%81%ED%83%9C%EA%B4%80%EB%A6%AC_%ED%86%B5%ED%95%A9_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
%%writefile reactWms/node-server/server.js

const express = require('express');
const cors = require('cors');
const mysql = require('mysql2/promise');
const path = require('path');

const app = express();
const PORT = 3000;

app.use(cors());
app.use(express.json());

// static 디렉토리 제공
app.use(express.static(path.join(__dirname, 'static')));

// DB 연결
const pool = mysql.createPool({
  host: 'localhost',
  user: 'testuser',
  password: '1234',
  database: 'testdb',
  waitForConnections: true,
  connectionLimit: 10,
  queueLimit: 0,
  dateStrings: true
});

function mapInventoryRow(row) {
  return {
    id: row.id,
    productInitial: row.product_initial,
    productName: row.product_name,
    category: row.category,
    status: row.status,
    location: row.location,
    storagePeriod: row.storage_period,
    inboundDate: row.inbound_date,
    outboundDueDate: row.outbound_due_date,
    expectedAmount: Number(row.expected_amount).toLocaleString('ko-KR') + '원'
  };
}

// 재고 목록 조회
app.get('/api/inventory', async function (req, res) {
  try {
    const [rows] = await pool.query(`
      SELECT *
      FROM inventory_items
      ORDER BY id DESC
    `);

    await new Promise(function (resolve) {
      setTimeout(resolve, 1500);
    });

    res.json(rows.map(mapInventoryRow));
  } catch (error) {
    res.status(500).json({ message: '재고 목록 조회 실패' });
  }
});

// 재고 등록
app.post('/api/inventory', async function (req, res) {
  try {
    const {
      productInitial,
      productName,
      category,
      status,
      location,
      storagePeriod,
      inboundDate,
      outboundDueDate,
      expectedAmount
    } = req.body;

    if (
      !productInitial ||
      !productName ||
      !category ||
      !status ||
      !location ||
      !storagePeriod ||
      !inboundDate ||
      !outboundDueDate ||
      !expectedAmount
    ) {
      return res.status(400).json({ message: '모든 항목을 입력해야 합니다.' });
    }

    if (Number(expectedAmount) >= 10000000) {
      return res.status(500).json({ message: '금액이 너무 커서 등록 실패' });
    }

    await pool.query(
      `
      INSERT INTO inventory_items
      (
        product_initial,
        product_name,
        category,
        status,
        location,
        storage_period,
        inbound_date,
        outbound_due_date,
        expected_amount
      )
      VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
      `,
      [
        productInitial,
        productName,
        category,
        status,
        location,
        storagePeriod,
        inboundDate,
        outboundDueDate,
        Number(expectedAmount)
      ]
    );

    res.json({ success: true });
  } catch (error) {
    res.status(500).json({ message: '재고 등록 실패' });
  }
});

// 메인 카드 조회
app.get('/api/summary-cards', async function (req, res) {
  try {
    const [[summary]] = await pool.query(`
      SELECT
        COUNT(*) AS totalCount,
        SUM(CASE WHEN status = '입고예정' THEN 1 ELSE 0 END) AS inboundPlanned,
        SUM(CASE WHEN status = '입고완료' THEN 1 ELSE 0 END) AS inboundDone,
        SUM(CASE WHEN status = '출고예정' THEN 1 ELSE 0 END) AS outboundPlanned,
        SUM(CASE WHEN status = '출고완료' THEN 1 ELSE 0 END) AS outboundDone,
        SUM(expected_amount) AS totalAmount
      FROM inventory_items
    `);

    res.json({
      main: [
        {
          title: '견적내역',
          value: String(summary.totalCount || 0),
          icon: '□'
        },
        {
          title: '입고내역',
          value: String((summary.inboundPlanned || 0) + (summary.inboundDone || 0)),
          subText: `입고예정 ${summary.inboundPlanned || 0} / 입고완료 ${summary.inboundDone || 0}`,
          icon: '≡'
        },
        {
          title: '출고내역',
          value: String((summary.outboundPlanned || 0) + (summary.outboundDone || 0)),
          subText: `출고예정 ${summary.outboundPlanned || 0} / 출고완료 ${summary.outboundDone || 0}`,
          icon: '◎'
        },
        {
          title: '정산내역',
          value: Number(summary.totalAmount || 0).toLocaleString('ko-KR'),
          icon: '◉'
        }
      ]
    });
  } catch (error) {
    res.status(500).json({ message: '카드 요약 조회 실패' });
  }
});

// React 라우팅 대응
app.get('/', function (req, res) {
  res.sendFile(path.join(__dirname, 'static', 'index.html'));
});

app.get('/*rest', function (req, res) {
  res.sendFile(path.join(__dirname, 'static', 'index.html'));
});

app.listen(PORT, function () {
  console.log('server running on 3000');
});

Overwriting reactWms/node-server/server.js


In [33]:
%%writefile reactWms/react/src/main.jsx

import React from 'react';
import ReactDOM from 'react-dom/client';
import { QueryClient, QueryClientProvider } from '@tanstack/react-query';
import App from './App';
import './style.css';

const queryClient = new QueryClient();

ReactDOM.createRoot(document.getElementById('root')).render(
  <React.StrictMode>
    <QueryClientProvider client={queryClient}>
      <App />
    </QueryClientProvider>
  </React.StrictMode>
);

Overwriting reactWms/react/src/main.jsx


In [34]:
%%writefile reactWms/react/src/App.jsx

import Dashboard from './pages/Dashboard';

function App() {
  return <Dashboard />;
}

export default App;

Overwriting reactWms/react/src/App.jsx


In [35]:
%%writefile reactWms/react/src/data/dashboardData.js

export const MENU_GROUPS = [
  {
    title: 'HOME',
    items: [
      { label: '메인페이지', icon: '□', type: 'main' }
    ]
  },
  {
    title: '입고 및 계약',
    items: [
      { label: '계약 현황', icon: '◻', type: 'sub' },
      { label: '입고 현황', icon: '◻', type: 'sub' }
    ]
  },
  {
    title: '출고',
    items: [
      { label: '출고 현황', icon: '○', type: 'sub' }
    ]
  },
  {
    title: '재고',
    items: [
      { label: '재고 현황', icon: '○', type: 'sub' }
    ]
  },
  {
    title: '창고관리',
    items: [
      { label: '기자재 관리', icon: '○', type: 'sub' }
    ]
  },
  {
    title: '시스템',
    items: [
      { label: '공지사항', icon: '•', type: 'sub' },
      { label: '문의사항', icon: '•', type: 'sub' },
      { label: '사원관리', icon: '•', type: 'sub' }
    ]
  }
];

export const SUMMARY_CARD_MAP = {
  '메인페이지': [
    { title: '견적내역', value: '0', icon: '□' },
    { title: '입고내역', value: '0', subText: '입고예정 0 / 입고완료 0', icon: '≡' },
    { title: '출고내역', value: '0', subText: '출고예정 0 / 출고완료 0', icon: '◎' },
    { title: '정산내역', value: '0', icon: '◉' }
  ]
};

Overwriting reactWms/react/src/data/dashboardData.js


In [46]:
%%writefile reactWms/react/src/hooks/useDashboardState.js

import { useMemo, useState } from 'react';
import { useMutation, useQuery, useQueryClient } from '@tanstack/react-query';
import { MENU_GROUPS, SUMMARY_CARD_MAP } from '../data/dashboardData';
import {
  createInventoryItem,
  fetchInventoryRows,
  fetchSummaryCards
} from '../api/inventoryApi';

function normalizeText(value) {
  return String(value).toLowerCase().trim();
}

export function useDashboardState() {
  const queryClient = useQueryClient();

  const [selectedMenu, setSelectedMenu] = useState('메인페이지');
  const [searchKeyword, setSearchKeyword] = useState('');

  const inventoryQuery = useQuery({
    queryKey: ['inventory'],
    queryFn: fetchInventoryRows,
    staleTime: 10000
  });

  const summaryQuery = useQuery({
    queryKey: ['summaryCards'],
    queryFn: fetchSummaryCards,
    staleTime: 10000
  });

  const mutation = useMutation({
    mutationFn: createInventoryItem,

    onMutate: async function (newItem) {
      await queryClient.cancelQueries({ queryKey: ['inventory'] });

      const previousInventory = queryClient.getQueryData(['inventory']);

      queryClient.setQueryData(['inventory'], function (old) {
        return [
          {
            id: Date.now(),
            ...newItem,
            expectedAmount:
              Number(newItem.expectedAmount).toLocaleString('ko-KR') + '원'
          },
          ...(old || [])
        ];
      });

      return { previousInventory };
    },

    onError: function (error, newItem, context) {
      if (context?.previousInventory) {
        queryClient.setQueryData(['inventory'], context.previousInventory);
      }
    },

    onSettled: function () {
      queryClient.invalidateQueries({ queryKey: ['inventory'] });
      queryClient.invalidateQueries({ queryKey: ['summaryCards'] });
    }
  });

  const summaryCards = useMemo(function () {
    if (selectedMenu !== '메인페이지') {
      return SUMMARY_CARD_MAP[selectedMenu] || SUMMARY_CARD_MAP['메인페이지'];
    }

    return summaryQuery.data?.main || SUMMARY_CARD_MAP['메인페이지'];
  }, [selectedMenu, summaryQuery.data]);

  const filteredInventoryRows = useMemo(function () {
    const rows = inventoryQuery.data || [];
    const keyword = normalizeText(searchKeyword);

    if (!keyword) {
      return rows;
    }

    return rows.filter(function (row) {
      return [
        row.productName,
        row.category,
        row.status,
        row.location,
        row.storagePeriod,
        row.inboundDate,
        row.outboundDueDate,
        row.expectedAmount
      ].some(function (field) {
        return normalizeText(field).includes(keyword);
      });
    });
  }, [inventoryQuery.data, searchKeyword]);

  return {
    menuGroups: MENU_GROUPS,
    selectedMenu,
    setSelectedMenu,
    searchKeyword,
    setSearchKeyword,
    summaryCards,
    filteredInventoryRows,
    inventoryQuery,
    summaryQuery,
    createItemMutation: mutation
  };
}

Overwriting reactWms/react/src/hooks/useDashboardState.js


In [47]:
%%writefile reactWms/react/src/api/inventoryApi.js

export async function fetchInventoryRows() {
  const response = await fetch('/api/inventory');

  if (!response.ok) {
    const errorData = await response.json();
    throw new Error(errorData.message || '재고 목록 조회 실패');
  }

  return await response.json();
}

export async function createInventoryItem(itemData) {
  const response = await fetch('/api/inventory', {
    method: 'POST',
    headers: {
      'Content-Type': 'application/json'
    },
    body: JSON.stringify(itemData)
  });

  if (!response.ok) {
    const errorData = await response.json();
    throw new Error(errorData.message || '재고 등록 실패');
  }

  return await response.json();
}

export async function fetchSummaryCards() {
  const response = await fetch('/api/summary-cards');

  if (!response.ok) {
    const errorData = await response.json();
    throw new Error(errorData.message || '요약 카드 조회 실패');
  }

  return await response.json();
}

Overwriting reactWms/react/src/api/inventoryApi.js


In [38]:
%%writefile reactWms/react/src/pages/Dashboard.jsx

import Sidebar from '../components/Sidebar';
import Topbar from '../components/Topbar';
import HeroBanner from '../components/HeroBanner';
import SummaryCard from '../components/SummaryCard';
import InventoryTable from '../components/InventoryTable';
import InventoryCreateForm from '../components/InventoryCreateForm';
import { useDashboardState } from '../hooks/useDashboardState';

function Dashboard() {
  const {
    menuGroups,
    selectedMenu,
    setSelectedMenu,
    searchKeyword,
    setSearchKeyword,
    summaryCards,
    filteredInventoryRows,
    inventoryQuery,
    createItemMutation
  } = useDashboardState();

  return (
    <div className="dashboard-layout">
      <Sidebar
        menuGroups={menuGroups}
        selectedMenu={selectedMenu}
        setSelectedMenu={setSelectedMenu}
      />

      <div className="dashboard-main">
        <Topbar
          searchKeyword={searchKeyword}
          setSearchKeyword={setSearchKeyword}
        />

        <div className="dashboard-content">
          <HeroBanner title={selectedMenu} />

          <div className="status-panel">
            {inventoryQuery.isLoading && (
              <p className="status-text">재고 목록을 최초 조회하는 중입니다.</p>
            )}

            {!inventoryQuery.isLoading && inventoryQuery.isFetching && (
              <p className="status-text">
                캐시된 데이터를 먼저 보여주고 최신 데이터를 다시 확인하는 중입니다.
              </p>
            )}

            {inventoryQuery.error && (
              <p className="status-text error-text">
                {inventoryQuery.error.message}
              </p>
            )}

            {createItemMutation.isPending && (
              <p className="status-text">재고 등록 요청을 처리하는 중입니다.</p>
            )}

            {createItemMutation.isError && (
              <p className="status-text error-text">
                {createItemMutation.error.message}
              </p>
            )}

            {createItemMutation.isSuccess && (
              <p className="status-text success-text">
                재고 등록이 완료되었고 서버 기준으로 다시 조회했습니다.
              </p>
            )}
          </div>

          <section className="summary-grid">
            {summaryCards.map(function (card) {
              return (
                <SummaryCard
                  key={card.title}
                  title={card.title}
                  value={card.value}
                  subText={card.subText}
                  icon={card.icon}
                />
              );
            })}
          </section>

          <InventoryCreateForm mutation={createItemMutation} />

          <InventoryTable rows={filteredInventoryRows} />
        </div>
      </div>
    </div>
  );
}

export default Dashboard;

Overwriting reactWms/react/src/pages/Dashboard.jsx


In [39]:
%%writefile reactWms/react/src/components/Sidebar.jsx

function Sidebar({ menuGroups, selectedMenu, setSelectedMenu }) {
  return (
    <aside className="sidebar">
      <div className="sidebar-logo-row">
        <div className="sidebar-logo-mark">
          <span className="mark-left"></span>
          <span className="mark-right"></span>
        </div>
        <h1 className="sidebar-logo-text">Smart Wms</h1>
      </div>

      <div className="sidebar-menu-wrap">
        {menuGroups.map(function (group) {
          return (
            <section className="sidebar-group" key={group.title}>
              <h2 className="sidebar-group-title">{group.title}</h2>

              {group.items.map(function (item) {
                const isActive = selectedMenu === item.label;
                const className =
                  item.type === 'main'
                    ? isActive
                      ? 'sidebar-menu active'
                      : 'sidebar-menu'
                    : isActive
                    ? 'sidebar-submenu active-submenu'
                    : 'sidebar-submenu';

                return (
                  <button
                    key={item.label}
                    type="button"
                    className={className}
                    onClick={function () {
                      setSelectedMenu(item.label);
                    }}
                  >
                    <span
                      className={
                        item.type === 'main'
                          ? 'sidebar-menu-icon'
                          : 'sidebar-submenu-icon'
                      }
                    >
                      {item.icon}
                    </span>
                    <span>{item.label}</span>
                  </button>
                );
              })}
            </section>
          );
        })}
      </div>
    </aside>
  );
}

export default Sidebar;

Overwriting reactWms/react/src/components/Sidebar.jsx


In [40]:
%%writefile reactWms/react/src/components/Topbar.jsx

function Topbar({ searchKeyword, setSearchKeyword }) {
  return (
    <header className="topbar">
      <div className="topbar-left">
        <button type="button" className="menu-toggle-button">
          ☰
        </button>

        <div className="topbar-search-box">
          <input
            type="text"
            placeholder="Search"
            value={searchKeyword}
            onChange={function (event) {
              setSearchKeyword(event.target.value);
            }}
          />
        </div>
      </div>

      <div className="topbar-right">
        <div className="topbar-circle"></div>
        <div className="topbar-avatar">●</div>
      </div>
    </header>
  );
}

export default Topbar;

Overwriting reactWms/react/src/components/Topbar.jsx


In [41]:
%%writefile reactWms/react/src/components/HeroBanner.jsx

function HeroBanner({ title }) {
  return (
    <section className="hero-banner">
      <h2 className="hero-title">{title}</h2>
      <button type="button" className="hero-button">
        Create New Project
      </button>
    </section>
  );
}

export default HeroBanner;

Overwriting reactWms/react/src/components/HeroBanner.jsx


In [42]:
%%writefile reactWms/react/src/components/SummaryCard.jsx

function SummaryCard({ title, value, subText, icon }) {
  return (
    <article className="summary-card">
      <div className="summary-card-top">
        <h3 className="summary-card-title">{title}</h3>
        <div className="summary-card-icon-box">{icon}</div>
      </div>

      <div className="summary-card-body">
        <p className="summary-card-value">{value}</p>
        {subText ? (
          <p className="summary-card-subtext">{subText}</p>
        ) : null}
      </div>
    </article>
  );
}

export default SummaryCard;

Overwriting reactWms/react/src/components/SummaryCard.jsx


In [43]:
%%writefile reactWms/react/src/components/InventoryTable.jsx

function InventoryTable({ rows }) {
  return (
    <section className="inventory-panel">
      <h3 className="inventory-panel-title">입/출고 내역</h3>

      <div className="inventory-table-wrap">
        <table className="inventory-table">
          <thead>
            <tr>
              <th>상품이름</th>
              <th>상품분류</th>
              <th>진행상태</th>
              <th>보관위치</th>
              <th>보관기간</th>
              <th>입고일</th>
              <th>출고예정일</th>
              <th>정산예정금액</th>
            </tr>
          </thead>

          <tbody>
            {rows.length > 0 ? (
              rows.map(function (row) {
                return (
                  <tr key={row.id}>
                    <td className="product-cell">
                      <div className="product-info">
                        <div className="product-logo">{row.productInitial}</div>
                        <span>{row.productName}</span>
                      </div>
                    </td>
                    <td>{row.category}</td>
                    <td>{row.status}</td>
                    <td>{row.location}</td>
                    <td>{row.storagePeriod}</td>
                    <td>{row.inboundDate}</td>
                    <td>{row.outboundDueDate}</td>
                    <td>{row.expectedAmount}</td>
                  </tr>
                );
              })
            ) : (
              <tr>
                <td colSpan="8" className="empty-row">
                  검색 결과가 없습니다.
                </td>
              </tr>
            )}
          </tbody>
        </table>
      </div>

      <div className="inventory-panel-footer">
        <a href="/">모든 내역 보기</a>
      </div>
    </section>
  );
}

export default InventoryTable;

Overwriting reactWms/react/src/components/InventoryTable.jsx


In [44]:
%%writefile reactWms/react/src/components/InventoryCreateForm.jsx

import { useState } from 'react';

function InventoryCreateForm({ mutation }) {
  const [form, setForm] = useState({
    productInitial: '',
    productName: '',
    category: '일반',
    status: '입고예정',
    location: '',
    storagePeriod: '',
    inboundDate: '',
    outboundDueDate: '',
    expectedAmount: ''
  });

  function handleChange(event) {
    setForm({
      ...form,
      [event.target.name]: event.target.value
    });
  }

  function handleSubmit(event) {
    event.preventDefault();

    mutation.mutate({
      ...form,
      expectedAmount: Number(form.expectedAmount)
    });

    setForm({
      productInitial: '',
      productName: '',
      category: '일반',
      status: '입고예정',
      location: '',
      storagePeriod: '',
      inboundDate: '',
      outboundDueDate: '',
      expectedAmount: ''
    });
  }

  return (
    <section className="inventory-panel create-form-panel">
      <h3 className="inventory-panel-title">재고 등록</h3>

      <form onSubmit={handleSubmit} className="inventory-create-form">
        <input
          name="productInitial"
          value={form.productInitial}
          onChange={handleChange}
          placeholder="상품 이니셜"
        />

        <input
          name="productName"
          value={form.productName}
          onChange={handleChange}
          placeholder="상품명"
        />

        <input
          name="location"
          value={form.location}
          onChange={handleChange}
          placeholder="보관 위치"
        />

        <input
          name="storagePeriod"
          value={form.storagePeriod}
          onChange={handleChange}
          placeholder="보관 기간"
        />

        <input
          type="date"
          name="inboundDate"
          value={form.inboundDate}
          onChange={handleChange}
        />

        <input
          type="date"
          name="outboundDueDate"
          value={form.outboundDueDate}
          onChange={handleChange}
        />

        <input
          type="number"
          name="expectedAmount"
          value={form.expectedAmount}
          onChange={handleChange}
          placeholder="정산 예정 금액"
        />

        <select
          name="category"
          value={form.category}
          onChange={handleChange}
        >
          <option value="일반">일반</option>
          <option value="냉장">냉장</option>
        </select>

        <select
          name="status"
          value={form.status}
          onChange={handleChange}
        >
          <option value="입고예정">입고예정</option>
          <option value="입고완료">입고완료</option>
          <option value="출고예정">출고예정</option>
          <option value="출고완료">출고완료</option>
        </select>

        <button type="submit" className="hero-button">
          재고 등록
        </button>
      </form>
    </section>
  );
}

export default InventoryCreateForm;


Overwriting reactWms/react/src/components/InventoryCreateForm.jsx


In [45]:
%%writefile reactWms/react/src/style.css

* {
  box-sizing: border-box;
  margin: 0;
  padding: 0;
}

html,
body,
#root {
  min-height: 100%;
  font-family: Arial, sans-serif;
  background: #f5f6fa;
  color: #2c3442;
}

button,
input,
select {
  font: inherit;
}

body {
  background: #f5f6fa;
}

.dashboard-layout {
  display: flex;
  min-height: 100vh;
}

.sidebar {
  width: 196px;
  min-width: 196px;
  background: #f6f6f7;
  border-right: 1px solid #e5e8ef;
  padding: 18px 18px 24px;
}

.sidebar-logo-row {
  display: flex;
  align-items: center;
  gap: 10px;
  margin-bottom: 34px;
}

.sidebar-logo-mark {
  display: flex;
  gap: 2px;
}

.mark-left,
.mark-right {
  display: block;
  width: 14px;
  height: 18px;
  border-radius: 6px;
}

.mark-left {
  background: #5d6cf8;
}

.mark-right {
  background: #53d0c4;
}

.sidebar-logo-text {
  font-size: 20px;
  font-weight: 700;
  color: #212631;
}

.sidebar-menu-wrap {
  display: flex;
  flex-direction: column;
  gap: 22px;
}

.sidebar-group-title {
  font-size: 11px;
  font-weight: 700;
  color: #333a46;
  margin-bottom: 12px;
}

.sidebar-menu,
.sidebar-submenu {
  border: none;
  background: transparent;
  width: 100%;
  text-align: left;
  cursor: pointer;
}

.sidebar-menu {
  display: flex;
  align-items: center;
  gap: 10px;
  height: 36px;
  border-radius: 7px;
  padding: 0 14px;
  font-size: 13px;
  color: #6d7789;
}

.sidebar-submenu {
  display: flex;
  align-items: center;
  gap: 10px;
  min-height: 30px;
  padding: 4px 10px 4px 14px;
  font-size: 13px;
  color: #6d7789;
  border-radius: 7px;
}

.sidebar-menu:hover,
.sidebar-submenu:hover {
  background: rgba(91, 75, 243, 0.08);
}

.sidebar-menu.active {
  background: #5b4bf3;
  color: #ffffff;
  box-shadow: 0 8px 16px rgba(91, 75, 243, 0.25);
}

.active-submenu {
  background: rgba(91, 75, 243, 0.12);
  color: #4f46e5;
  font-weight: 600;
}

.sidebar-menu:focus,
.sidebar-submenu:focus {
  outline: 2px solid rgba(91, 75, 243, 0.25);
  outline-offset: 2px;
}

.sidebar-menu-icon,
.sidebar-submenu-icon {
  width: 14px;
  text-align: center;
  font-size: 11px;
}

.dashboard-main {
  flex: 1;
  min-width: 0;
  display: flex;
  flex-direction: column;
}

.topbar {
  height: 49px;
  background: #ffffff;
  border-bottom: 1px solid #e8ebf1;
  display: flex;
  align-items: center;
  justify-content: space-between;
  padding: 0 22px 0 18px;
}

.topbar-left {
  display: flex;
  align-items: center;
  gap: 12px;
}

.menu-toggle-button {
  border: none;
  background: transparent;
  color: #616b7b;
  font-size: 16px;
  cursor: pointer;
}

.topbar-search-box {
  width: 218px;
  height: 31px;
  border: 1px solid #dde2ea;
  border-radius: 5px;
  background: #fafbfc;
  display: flex;
  align-items: center;
  padding: 0 14px;
}

.topbar-search-box input {
  width: 100%;
  border: none;
  outline: none;
  background: transparent;
  font-size: 13px;
  color: #5d6677;
}

.topbar-right {
  display: flex;
  align-items: center;
  gap: 12px;
}

.topbar-circle {
  width: 33px;
  height: 33px;
  border-radius: 50%;
  background: #f1f2f6;
  border: 1px solid #edf0f4;
}

.topbar-avatar {
  width: 35px;
  height: 35px;
  border-radius: 50%;
  background: #18202d;
  color: #ffffff;
  display: flex;
  align-items: center;
  justify-content: center;
  font-size: 12px;
}

.dashboard-content {
  padding: 0 21px 28px;
}

.hero-banner {
  height: 169px;
  background: #5b4bf3;
  display: flex;
  align-items: flex-start;
  justify-content: space-between;
  padding: 46px 22px 0;
  color: #ffffff;
}

.hero-title {
  font-size: 25px;
  font-weight: 500;
}

.hero-button {
  width: 138px;
  height: 33px;
  border: none;
  border-radius: 6px;
  background: #ffffff;
  color: #1f2430;
  font-size: 13px;
  cursor: pointer;
}

.summary-grid {
  margin-top: -68px;
  position: relative;
  z-index: 2;
  display: grid;
  grid-template-columns: repeat(4, 1fr);
  gap: 16px;
  padding: 0 2px;
}

.summary-card {
  background: #ffffff;
  border-radius: 10px;
  min-height: 134px;
  padding: 16px;
  border: 1px solid #eceef4;
  box-shadow: 0 10px 18px rgba(32, 43, 67, 0.08);
}

.summary-card-top {
  display: flex;
  align-items: flex-start;
  justify-content: space-between;
  margin-bottom: 10px;
}

.summary-card-title {
  font-size: 14px;
  font-weight: 600;
  color: #2c3442;
  margin-top: 6px;
}

.summary-card-icon-box {
  width: 33px;
  height: 33px;
  border-radius: 6px;
  background: #ece8ff;
  color: #6f63ec;
  display: flex;
  align-items: center;
  justify-content: center;
  font-size: 14px;
}

.summary-card-value {
  font-size: 31px;
  font-weight: 700;
  color: #1f2937;
  margin-top: 8px;
}

.summary-card-subtext {
  margin-top: 8px;
  font-size: 12px;
  color: #8a95a8;
}

.inventory-panel {
  margin-top: 22px;
  background: #ffffff;
  border-radius: 10px;
  border: 1px solid #eceef3;
  box-shadow: 0 10px 18px rgba(32, 43, 67, 0.07);
  overflow: hidden;
}

.inventory-panel-title {
  padding: 16px 18px;
  font-size: 15px;
  font-weight: 600;
  color: #2e3544;
  border-bottom: 1px solid #eef1f5;
}

.inventory-table-wrap {
  width: 100%;
  overflow-x: auto;
}

.inventory-table {
  width: 100%;
  border-collapse: collapse;
  min-width: 1000px;
}

.inventory-table thead th {
  background: #f3f6fa;
  color: #7b8799;
  font-size: 12px;
  font-weight: 500;
  text-align: left;
  padding: 11px 18px;
  border-bottom: 1px solid #e9edf3;
}

.inventory-table tbody td {
  padding: 16px 18px;
  border-bottom: 1px solid #eef1f5;
  color: #677285;
  font-size: 13px;
}

.product-info {
  display: flex;
  align-items: center;
  gap: 14px;
  color: #2d3443;
}

.product-logo {
  width: 30px;
  height: 30px;
  border-radius: 4px;
  border: 1px solid #e6e9ef;
  display: flex;
  align-items: center;
  justify-content: center;
  background: #ffffff;
  color: #5b4bf3;
  font-weight: 700;
}

.empty-row {
  text-align: center;
  color: #8a95a8;
}

.inventory-panel-footer {
  height: 38px;
  display: flex;
  align-items: center;
  justify-content: center;
}

.inventory-panel-footer a {
  color: #5e52f3;
  text-decoration: none;
  font-size: 13px;
  font-weight: 500;
}

.status-panel {
  margin-top: 16px;
  background: #ffffff;
  border: 1px solid #eceef3;
  border-radius: 10px;
  padding: 12px 16px;
}

.status-text {
  font-size: 13px;
  color: #495468;
  margin-bottom: 6px;
}

.status-text:last-child {
  margin-bottom: 0;
}

.error-text {
  color: #c62828;
}

.success-text {
  color: #2e7d32;
}

.create-form-panel {
  padding-bottom: 16px;
}

.inventory-create-form {
  display: grid;
  grid-template-columns: repeat(3, 1fr);
  gap: 12px;
  padding: 16px 18px 0;
}

.inventory-create-form input,
.inventory-create-form select {
  height: 38px;
  border: 1px solid #dde2ea;
  border-radius: 6px;
  padding: 0 12px;
  background: #ffffff;
}

.inventory-create-form button {
  align-self: center;
}

@media (max-width: 1200px) {
  .summary-grid {
    grid-template-columns: repeat(2, 1fr);
  }

  .inventory-create-form {
    grid-template-columns: repeat(2, 1fr);
  }
}

@media (max-width: 900px) {
  .sidebar {
    display: none;
  }

  .summary-grid {
    grid-template-columns: 1fr;
    margin-top: 16px;
  }

  .hero-banner {
    height: auto;
    padding: 24px 18px;
    flex-direction: column;
    gap: 16px;
  }

  .inventory-create-form {
    grid-template-columns: 1fr;
  }
}

Overwriting reactWms/react/src/style.css
